In [0]:
import requests, csv, os
from io import StringIO

maryland_data_url = "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=MD&years=2024&actions_taken=1,3"
response = requests.get(maryland_data_url)
response.raise_for_status()

maryland_data_str = response.content.decode("utf-8")

In [0]:
# records = list(csv.DictReader(StringIO(maryland_data_str)))
# maryland_dataset = spark.createDataFrame(records)

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

vol_csv_path = '/Volumes/mortgage-lending-risk-and-regulatory-analytics/default/api_landing/source_csv'
file_path = f"{vol_csv_path}/api_data_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}_{uuid4().hex[:8]}.csv"

os.makedirs(vol_csv_path, exist_ok=True)

with open(file_path, "w", encoding="utf-8", newline="") as f:
    f.write(maryland_data_str)

print(f"File saved: {file_path}")


In [0]:
file_path = "/Volumes/mortgage-lending-risk-and-regulatory-analytics/default/api_landing/source_csv/api_data_20261010_225019_61e3e547.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(file_path)
)


print("Total records:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()

In [0]:
df.select(
    "activity_year",
    "lei",
    "state_code",
    "action_taken",
    "loan_amount",
    "debt_to_income_ratio",
    "interest_rate"
).show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

DB = "`mortgage-lending-risk-and-regulatory-analytics`.default"

bronze_df = (
    df.withColumn(
        "source_file",
        F.lit("hmda_2024_md.csv")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{DB}.hmda_bronze_2024_md"
    )
)

In [0]:
DB = "`mortgage-lending-risk-and-regulatory-analytics`.default"

bronze = spark.table(
    f"{DB}.hmda_bronze_2024_md"
)

display(
    bronze.select(
        "lei",
        "action_taken",
        "loan_amount",
        "interest_rate",
        "debt_to_income_ratio",
        "loan_to_value_ratio"
    ).limit(20)
)

display(
    bronze.groupBy("action_taken").count()
)


In [0]:
bronze.select("derived_loan_product_type").distinct().show()

In [0]:
enum_dict = {}

for col in bronze.columns:
    col_val = [v[0] for v in bronze.select(col).distinct().collect()]
    if len(col_val) < 20:
        print(col, len(col_val))
        enum_dict[col] = col_val


In [0]:
enum_dict

import json

with open("metadata.json", "w") as f:
    json.dump(enum_dict, f)



In [0]:
del enum_dict["ingestion_timestamp"]

In [0]:
import requests

page = requests.get("https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields/#derived_loan_product_type$0")

In [0]:
# page.content.decode("utf-8")

with open("to_del.html", "w") as f:
    f.writelines(page.content.decode("utf-8"))

In [0]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page.content.decode("utf-8"), 'html.parser')

# Extract data using tag names, attributes, or CSS selectors
print("H1 Text:", soup.find('h3'))